# 08 — Solution Generation

This notebook demonstrates the **Solution Generation Workflow** — a sequential pipeline that produces structured exam answers with depth scaled by mark allocation.

## Pipeline
1. **Retrieve** — Fetch relevant context from the knowledge base
2. **Generate** — Use LLM with marks-based depth control
3. **Format** — Validate output as a structured `Solution` model

## Depth Scaling
| Marks | Depth | Description |
|-------|-------|-------------|
| 1-3 | Brief | 2-3 sentence answer |
| 4-6 | Moderate | Key points with short paragraphs |
| 7+ | Detailed | Full explanation, examples, diagram hints |

In [ ]:
import sys
sys.path.insert(0, '..')

from src.llm import LLMClient
from src.workflows.solutions import SolutionWorkflow, _get_depth_category, DEPTH_PROMPTS
from models.output import Solution

print("Imports successful!")

## 1. Initialize the Workflow

The `SolutionWorkflow` takes an `LLMClient` and an optional `Retriever`. For this demo we'll use the LLM client directly (no retriever since the vector store may not be set up yet).

In [ ]:
# Initialize LLM client (uses .env API keys)
llm_client = LLMClient()
print(f"Available providers: {[p.value for p in llm_client.available_providers]}")

# Create workflow without retriever for demo
workflow = SolutionWorkflow(llm_client=llm_client, retriever=None)
print("SolutionWorkflow initialized (no retriever — LLM-only mode)")

## 2. Depth Categories

The marks → depth mapping determines how detailed the generated answer will be.

In [ ]:
# Show depth mapping for various marks
test_marks = [1, 2, 3, 4, 5, 6, 7, 10, 15, 20]
print("Marks → Depth Category:")
print("-" * 30)
for m in test_marks:
    category = _get_depth_category(m)
    print(f"  {m:3d} marks → {category}")

print("\nDepth Instructions:")
print("=" * 50)
for category, instruction in DEPTH_PROMPTS.items():
    print(f"\n[{category.upper()}]")
    print(f"  {instruction}")

## 3. Generate a 2-Mark Brief Answer

Brief answers should be 2-3 sentences covering just the core definition.

In [ ]:
# 2-mark brief answer
solution_2 = workflow.generate(
    question="Define osmosis.",
    topic="Biology",
    marks=2,
)

print(f"Question: {solution_2.question}")
print(f"Marks: {solution_2.marks}")
print(f"Topic: {solution_2.topic}")
print(f"\nAnswer:\n{solution_2.answer}")
print(f"\nMarking Scheme:")
for item in solution_2.marking_scheme:
    print(f"  • {item}")
print(f"\nKey Points:")
for point in solution_2.key_points:
    print(f"  ✓ {point}")

## 4. Generate a 5-Mark Moderate Answer

Moderate answers include structured key points and a marking scheme.

In [ ]:
# 5-mark moderate answer
solution_5 = workflow.generate(
    question="Explain the process of photosynthesis and its importance.",
    topic="Biology",
    marks=5,
)

print(f"Question: {solution_5.question}")
print(f"Marks: {solution_5.marks}")
print(f"Topic: {solution_5.topic}")
print(f"\nAnswer:\n{solution_5.answer}")
print(f"\nMarking Scheme ({len(solution_5.marking_scheme)} items):")
for item in solution_5.marking_scheme:
    print(f"  • {item}")
print(f"\nKey Points ({len(solution_5.key_points)} points):")
for point in solution_5.key_points:
    print(f"  ✓ {point}")

## 5. Generate a 10-Mark Detailed Answer

Detailed answers include thorough explanations, examples, diagram hints, and comprehensive marking schemes.

In [ ]:
# 10-mark detailed answer
solution_10 = workflow.generate(
    question="Discuss the causes and consequences of World War I. Include political, economic, and social factors.",
    topic="History",
    marks=10,
)

print(f"Question: {solution_10.question}")
print(f"Marks: {solution_10.marks}")
print(f"Topic: {solution_10.topic}")
print(f"\nAnswer:\n{solution_10.answer}")
print(f"\nMarking Scheme ({len(solution_10.marking_scheme)} items):")
for item in solution_10.marking_scheme:
    print(f"  • {item}")
print(f"\nKey Points ({len(solution_10.key_points)} points):")
for point in solution_10.key_points:
    print(f"  ✓ {point}")

## 6. Compare Answer Lengths Across Marks

Let's compare how the same topic scales with different mark allocations.

In [ ]:
# Compare the three solutions
solutions = {
    "2-mark (brief)": solution_2,
    "5-mark (moderate)": solution_5,
    "10-mark (detailed)": solution_10,
}

print("Solution Comparison:")
print("=" * 60)
print(f"{'Level':<20} {'Answer Len':<12} {'Scheme Items':<14} {'Key Points'}")
print("-" * 60)
for label, sol in solutions.items():
    print(
        f"{label:<20} {len(sol.answer):<12} "
        f"{len(sol.marking_scheme):<14} {len(sol.key_points)}"
    )

## 7. Solution Model Validation

The `Solution` Pydantic model enforces data integrity.

In [ ]:
# Demonstrate the Solution model
from pydantic import ValidationError

# Valid solution
valid = Solution(
    question="What is gravity?",
    marks=3,
    answer="Gravity is a fundamental force of attraction between objects with mass.",
    marking_scheme=["1 mark: fundamental force", "1 mark: between masses", "1 mark: proportional to mass"],
    key_points=["Fundamental force", "Universal attraction", "F = Gm1m2/r^2"],
    topic="Physics",
)
print(f"Valid Solution: {valid.question} ({valid.marks} marks)")
print(f"  Answer length: {len(valid.answer)} chars")

# Invalid: marks out of range
try:
    Solution(
        question="Test",
        marks=0,  # Invalid: must be 1-100
        answer="Test answer",
        marking_scheme=[],
        key_points=[],
        topic="Test",
    )
except ValidationError as e:
    print(f"\nValidation Error (marks=0): {e.errors()[0]['msg']}")

# Serialization
print(f"\nJSON output (first 200 chars):\n{valid.to_json()[:200]}...")

## 8. Error Handling

The workflow validates inputs and handles failures gracefully.

In [ ]:
# Test invalid marks
try:
    workflow.generate(question="Test", topic="Test", marks=0)
except ValueError as e:
    print(f"Caught ValueError: {e}")

try:
    workflow.generate(question="Test", topic="Test", marks=101)
except ValueError as e:
    print(f"Caught ValueError: {e}")

## Summary

The Solution Generation Workflow provides:

- **Marks-based depth scaling**: Automatic adjustment of answer detail (brief → moderate → detailed)
- **Structured output**: Every solution includes answer text, marking scheme, and key points
- **Pydantic validation**: Output is type-safe and serializable
- **Retriever integration**: Can use context from the knowledge base when available
- **Graceful degradation**: Works without retriever, handles retrieval failures

### Next Steps
- Connect a populated vector store via `Retriever` for context-grounded answers
- Extend to support multi-part questions
- Add diagram generation hints for visual learners